In [25]:
!git clone https://github.com/boracandan/Algoverse_Truth_Directions_Research.git

fatal: destination path 'Algoverse_Truth_Directions_Research' already exists and is not an empty directory.


In [24]:
from ast import literal_eval
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import pickle
import gc
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import roc_auc_score

print("imports complete")

imports complete


In [26]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
save_dir = "/content/drive/MyDrive/algoverse_results"
DATA_DIR = "/content/Algoverse_Truth_Directions_Research/CoT_datasets/sentence_based_lexically_cleaned"
os.makedirs(save_dir, exist_ok=True)
csv_path = f"{save_dir}/results_database.csv"

In [5]:
gc.collect()
torch.cuda.empty_cache()

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Llama-8B", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"


model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    dtype=torch.float16,
    device_map="cuda",
    trust_remote_code=True,
)
model.eval()

print(f"✓ Model loaded")
print(f"  Device: {model.device}")
print(f"  GPU memory: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"  Layers: {model.config.num_hidden_layers}")

Loading tokenizer...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✓ Model loaded
  Device: cuda:0
  GPU memory: 16.06 GB
  Layers: 32


In [28]:
print(f"Loading sentence_based_lexically_cleaned datasets from {DATA_DIR}...")

# Load all task datasets
F0_train = pd.read_csv(f"{DATA_DIR}/F0_train.csv")
F0_test = pd.read_csv(f"{DATA_DIR}/F0_test.csv")
F1_train = pd.read_csv(f"{DATA_DIR}/F1_train.csv")
F1_test = pd.read_csv(f"{DATA_DIR}/F1_test.csv")
F2_train = pd.read_csv(f"{DATA_DIR}/F2_train.csv")
F2_test = pd.read_csv(f"{DATA_DIR}/F2_test.csv")
F3_train = pd.read_csv(f"{DATA_DIR}/F3_train.csv")
F3_test = pd.read_csv(f"{DATA_DIR}/F3_test.csv")
F4_train = pd.read_csv(f"{DATA_DIR}/F4_train.csv")
F4_test = pd.read_csv(f"{DATA_DIR}/F4_test.csv")
F5_train = pd.read_csv(f"{DATA_DIR}/F5_train.csv")
F5_test = pd.read_csv(f"{DATA_DIR}/F5_test.csv")
A1_train = pd.read_csv(f"{DATA_DIR}/A1_train.csv")
A1_test = pd.read_csv(f"{DATA_DIR}/A1_test.csv")
A2_train = pd.read_csv(f"{DATA_DIR}/A2_train.csv")
A2_test = pd.read_csv(f"{DATA_DIR}/A2_test.csv")
A3_train = pd.read_csv(f"{DATA_DIR}/A3_train.csv")
A3_test = pd.read_csv(f"{DATA_DIR}/A3_test.csv")

tasks_dict = {
    "F0": (F0_train, F0_test), "F1": (F1_train, F1_test), "F2": (F2_train, F2_test),
    "F3": (F3_train, F3_test), "F4": (F4_train, F4_test), "F5": (F5_train, F5_test),
    "A1": (A1_train, A1_test), "A2": (A2_train, A2_test), "A3": (A3_train, A3_test),
}
task_names = ["A1", "A2", "A3", "F0", "F1", "F2", "F3", "F4", "F5"]

print("✓ All sentence_based_lexically_cleaned datasets loaded:")
for name, (tr, te) in tasks_dict.items():
    print(f"    {name}: {len(tr)} train / {len(te)} test")

Loading sentence_based_lexically_cleaned datasets from /content/Algoverse_Truth_Directions_Research/CoT_datasets/sentence_based_lexically_cleaned...
✓ All sentence_based_lexically_cleaned datasets loaded:
    F0: 999 train / 464 test
    F1: 1100 train / 491 test
    F2: 1033 train / 464 test
    F3: 1008 train / 478 test
    F4: 1135 train / 568 test
    F5: 1216 train / 562 test
    A1: 675 train / 297 test
    A2: 673 train / 286 test
    A3: 692 train / 300 test


In [29]:
def decode_ids(id_column):
    return [
        tokenizer.decode(literal_eval(ids) if isinstance(ids, str) else ids)
        for ids in id_column
    ]


def activations_all_layers(model, statements, batch_size=4):
    statements = list(statements)
    num_layers = model.config.num_hidden_layers + 1
    activations_by_layer = [[] for _ in range(num_layers)]

    for i in range(0, len(statements), batch_size):
        batch = statements[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        for layer_idx, layer_hidden in enumerate(outputs.hidden_states):
            final_token = layer_hidden[:, -1, :]
            activations_by_layer[layer_idx].append(final_token.cpu())

        del outputs, inputs
        torch.cuda.empty_cache()

    return [torch.cat(layer_acts, dim=0) for layer_acts in activations_by_layer]


def train_probe(activations, labels, device='cuda'):
    (X_train, X_test), (y_train, y_test) = activations, labels
    X_train = torch.stack(list(X_train)).float().numpy()
    X_test = torch.stack(list(X_test)).float().numpy()
    y_train = y_train.to_numpy()
    y_test = y_test.to_numpy()

    train_mean = X_train.mean(axis=0)
    X_train = X_train - train_mean
    X_test = X_test - train_mean

    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

    hidden_dim = X_train.shape[1]
    probe = nn.Linear(hidden_dim, 1, bias=False).to(device)
    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=0.1)
    loss_fn = nn.BCEWithLogitsLoss()

    for _ in range(1000):
        optimizer.zero_grad()
        loss = loss_fn(probe(X_train_t).squeeze(-1), y_train_t)
        loss.backward()
        optimizer.step()

    probe.eval()
    with torch.no_grad():
        test_logits = probe(X_test_t).squeeze(-1).cpu().numpy()

    auroc = roc_auc_score(y_test, test_logits)
    w = probe.weight.detach().cpu().numpy().flatten()
    return w, train_mean, auroc


def train_all_layers(train_acts, test_acts, y_train, y_test):
    num_layers = len(train_acts)
    layer_results = {}
    for layer_idx in range(num_layers):
        X_train = train_acts[layer_idx]
        X_test = test_acts[layer_idx]
        X_train_np = torch.stack(list(X_train)).float().numpy()
        variance = X_train_np.var(axis=0) + 1e-6
        w, train_mean, auroc = train_probe((X_train, X_test), (y_train, y_test))
        layer_results[layer_idx] = {
            "auroc": auroc,
            "weights": w,
            "variance": variance,
            "train_mean": train_mean,
        }
        if layer_idx % 8 == 0:
            print(f"    Layer {layer_idx}: AUROC = {auroc:.4f}")
    return layer_results


def write_indomain_rows(task_name, layer_results, csv_path, condition="sentence-based-CoT"):
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        mask = ~((df["train_task"] == task_name) & 
                 (df["test_task"] == task_name) & 
                 (df["train_condition"] == condition) &
                 (df["model"] == "deepseek-r1-distill-8b"))
        df = df[mask]
    else:
        df = pd.DataFrame(columns=["train_task","test_task","train_condition","test_condition","layer","model","auroc"])

    new_rows = []
    for layer_idx, data in layer_results.items():
        new_rows.append({
            "train_task": task_name,
            "test_task": task_name,
            "train_condition": condition,
            "test_condition": condition,
            "layer": layer_idx,
            "model": "deepseek-r1-distill-8b",
            "auroc": data["auroc"],
        })

    df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    df.to_csv(csv_path, index=False)
    print(f"  ✓ Saved {len(new_rows)} layers for {task_name} ({condition})")

print("✓ All functions defined")

✓ All functions defined


In [30]:

task_name = "A1"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_A1 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_A1, csv_path, condition="sentence-based-CoT")


Processing A1...
    Layer 0: AUROC = 0.5374
    Layer 8: AUROC = 0.9891
    Layer 16: AUROC = 0.9984
    Layer 24: AUROC = 0.9980
    Layer 32: AUROC = 0.9944
  ✓ Saved 33 layers for A1 (sentence-based-CoT)


In [31]:

task_name = "A2"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_A2 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_A2, csv_path, condition="sentence-based-CoT")


Processing A2...
    Layer 0: AUROC = 0.5192
    Layer 8: AUROC = 0.9915
    Layer 16: AUROC = 1.0000
    Layer 24: AUROC = 1.0000
    Layer 32: AUROC = 0.9999
  ✓ Saved 33 layers for A2 (sentence-based-CoT)


In [32]:

task_name = "A3"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_A3 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_A3, csv_path, condition="sentence-based-CoT")


Processing A3...
    Layer 0: AUROC = 0.5567
    Layer 8: AUROC = 0.9907
    Layer 16: AUROC = 0.9999
    Layer 24: AUROC = 1.0000
    Layer 32: AUROC = 1.0000
  ✓ Saved 33 layers for A3 (sentence-based-CoT)


In [33]:
task_name = "F0"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F0 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F0, csv_path, condition="sentence-based-CoT")


Processing F0...
    Layer 0: AUROC = 0.5597
    Layer 8: AUROC = 0.8930
    Layer 16: AUROC = 0.9611
    Layer 24: AUROC = 0.9647
    Layer 32: AUROC = 0.9651
  ✓ Saved 33 layers for F0 (sentence-based-CoT)


In [34]:

task_name = "F1"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F1 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F1, csv_path, condition="sentence-based-CoT")


Processing F1...
    Layer 0: AUROC = 0.5046
    Layer 8: AUROC = 0.8994
    Layer 16: AUROC = 0.9750
    Layer 24: AUROC = 0.9758
    Layer 32: AUROC = 0.9801
  ✓ Saved 33 layers for F1 (sentence-based-CoT)


In [35]:

task_name = "F2"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F2 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F2, csv_path, condition="sentence-based-CoT")


Processing F2...
    Layer 0: AUROC = 0.5730
    Layer 8: AUROC = 0.8874
    Layer 16: AUROC = 0.9784
    Layer 24: AUROC = 0.9831
    Layer 32: AUROC = 0.9822
  ✓ Saved 33 layers for F2 (sentence-based-CoT)


In [36]:

task_name = "F3"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F3 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F3, csv_path, condition="sentence-based-CoT")


Processing F3...
    Layer 0: AUROC = 0.5458
    Layer 8: AUROC = 0.8660
    Layer 16: AUROC = 0.9515
    Layer 24: AUROC = 0.9571
    Layer 32: AUROC = 0.9535
  ✓ Saved 33 layers for F3 (sentence-based-CoT)


In [37]:

task_name = "F4"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F4 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F4, csv_path, condition="sentence-based-CoT")


Processing F4...
    Layer 0: AUROC = 0.5430
    Layer 8: AUROC = 0.8568
    Layer 16: AUROC = 0.9553
    Layer 24: AUROC = 0.9633
    Layer 32: AUROC = 0.9606
  ✓ Saved 33 layers for F4 (sentence-based-CoT)


In [38]:

task_name = "F5"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F5 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F5, csv_path, condition="sentence-based-CoT")


Processing F5...
    Layer 0: AUROC = 0.5603
    Layer 8: AUROC = 0.8963
    Layer 16: AUROC = 0.9693
    Layer 24: AUROC = 0.9715
    Layer 32: AUROC = 0.9730
  ✓ Saved 33 layers for F5 (sentence-based-CoT)


In [39]:
print("Pre-extracting test activations for all tasks...")
test_activations = {}

for task_name in task_names:
    print(f"  Extracting {task_name}...")
    _, test_df = tasks_dict[task_name]
    acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
    test_activations[task_name] = [
        layer_acts.float().numpy() if isinstance(layer_acts, torch.Tensor)
        else torch.stack(list(layer_acts)).float().numpy()
        for layer_acts in acts
    ]
    del acts
    torch.cuda.empty_cache()

print("\n All test activations extracted")

Pre-extracting test activations for all tasks...
  Extracting A1...
  Extracting A2...
  Extracting A3...
  Extracting F0...
  Extracting F1...
  Extracting F2...
  Extracting F3...
  Extracting F4...
  Extracting F5...

 All test activations extracted


In [40]:
all_probes = {
    "A1": results_A1, "A2": results_A2, "A3": results_A3,
    "F0": results_F0, "F1": results_F1, "F2": results_F2,
    "F3": results_F3, "F4": results_F4, "F5": results_F5,
}

df = pd.read_csv(csv_path)
mask = ~(
    (df["train_task"] != df["test_task"]) &
    (df["model"] == "deepseek-r1-distill-8b") &
    (df["train_condition"] == "sentence-based-CoT")
)
df = df[mask]

num_layers = model.config.num_hidden_layers + 1
cross_task_rows = []
total = len(task_names) * (len(task_names) - 1) * num_layers
completed = 0

for train_task in task_names:
    for test_task in task_names:
        if train_task == test_task:
            continue

        _, test_df = tasks_dict[test_task]
        test_labels = test_df["label"].to_numpy()

        for layer_idx in range(num_layers):
            w = all_probes[train_task][layer_idx]["weights"]
            mean_train = all_probes[train_task][layer_idx]["train_mean"]

            X_test = test_activations[test_task][layer_idx]
            X_test_centered = X_test - mean_train
            logits = X_test_centered @ w
            auroc = roc_auc_score(test_labels, logits)

            cross_task_rows.append({
                "train_task": train_task,
                "test_task": test_task,
                "train_condition": "sentence-based-CoT",
                "test_condition": "sentence-based-CoT",
                "layer": layer_idx,
                "model": "deepseek-r1-distill-8b",
                "auroc": auroc,
            })

            completed += 1
            if completed % 500 == 0:
                print(f"  {completed}/{total} done...")

df = pd.concat([df, pd.DataFrame(cross_task_rows)], ignore_index=True)
df.to_csv(csv_path, index=False)

print(f"\n✓ Done. Total rows: {len(df)}")
print(f"  In-domain: {len(df[df['train_task'] == df['test_task']])}")
print(f"  Cross-task: {len(df[df['train_task'] != df['test_task']])}")

  500/2376 done...
  1000/2376 done...
  1500/2376 done...
  2000/2376 done...

✓ Done. Total rows: 5346
  In-domain: 594
  Cross-task: 4752


In [2]:
import pandas as pd

csv_path = "/Users/Syam/truth_directions/Algoverse_Truth_Directions_Research/experiments/results_database.csv"
df = pd.read_csv(csv_path)

print("=" * 60)
print("CSV VALIDATION")
print("=" * 60)

print(f"\nTotal rows: {len(df)}")

print(f"\nRows by condition:")
conditions = df.groupby('train_condition').size()
print(conditions)

print(f"\nRows by model:")
models = df.groupby('model').size()
print(models)

print(f"\nIn-domain rows (train_task == test_task):")
indomain = df[df['train_task'] == df['test_task']]
print(f"Total: {len(indomain)}")
print(indomain.groupby('train_condition').size())

print(f"\nCross-task rows (train_task != test_task):")
crosstask = df[df['train_task'] != df['test_task']]
print(f"Total: {len(crosstask)}")
print(crosstask.groupby('train_condition').size())

print(f"\nLayer 25 AUROC by condition:")
layer25 = df[df['layer'] == 25].pivot_table(
    index='train_task',
    columns='train_condition',
    values='auroc'
)
print(layer25.round(4))

print(f"\n✓ Validation complete")

CSV VALIDATION

Total rows: 8019

Rows by condition:
train_condition
cot-zero-shot         2673
no-prompt             2673
sentence-based-CoT    2673
dtype: int64

Rows by model:
model
deepseek-r1-distill-8b    8019
dtype: int64

In-domain rows (train_task == test_task):
Total: 891
train_condition
cot-zero-shot         297
no-prompt             297
sentence-based-CoT    297
dtype: int64

Cross-task rows (train_task != test_task):
Total: 7128
train_condition
cot-zero-shot         2376
no-prompt             2376
sentence-based-CoT    2376
dtype: int64

Layer 25 AUROC by condition:
train_condition  cot-zero-shot  no-prompt  sentence-based-CoT
train_task                                                   
A1                      0.9391     0.5899              0.9391
A2                      0.9345     0.5503              0.9345
A3                      0.9346     0.5669              0.9346
F0                      0.9481     0.6339              0.9481
F1                      0.9394     0.6201 